In [11]:
from pathlib import Path
from dataclasses import dataclass

from dotenv import load_dotenv

from langchain_community.utilities import SQLDatabase
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.runtime import get_runtime


load_dotenv()
BASE_DIR = Path('C:/Users/User/Documents/Repositories/personal_ai_running_coach')

In [30]:
# ---- Database ----
# Define SQL database, and context for agent
DB_PATH = BASE_DIR / "data" / "strava.db"
db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")


@dataclass
class RuntimeContext:
    db: SQLDatabase

SUMMARY_PATH = BASE_DIR / "schema_summary.txt"
with open(SUMMARY_PATH, "r") as f:
    DB_SCHEMA = f.read()


In [7]:
# ---- Tools for agent ----
@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite SELECT query and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [ ]:
# --- System prompt ----
SYSTEM_PROMPT = f"""
You are an expert running coach and careful SQLite analyst.

You have access to a SQLite database of running activities.

DATABASE SCHEMA:
{DB_SCHEMA}

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read only; NEVER modify the database (no INSERT/UPDATE/DELETE/etc).
- Always LIMIT results to 5 unless explicitly needed.
- Prefer explicit column names (avoid SELECT *).
- If the tool returns 'Error:', fix your SQL and retry.
- When you know the correct SQL query, do not call execute_sql more than once per question.

Additional rules on interpreting requests and data:
- Use the schema to understand:
  - distance is in km
  - duration is in minutes
  - pace is in min/km
- The data is from Strava, and distances and paces may not be exact. Apply a degree of tolerance - e.g. if asked to look at 5K race performance, allow for distance between 4.8 and 5.2.

Coaching rules:
- Use past runs to support your reasoning.
- Look for trends (volume, HR drift, pacing).
- Be specific and actionable.
"""

In [34]:
# --- Define agent ---
agent = create_agent(
    model="openai:gpt-4o-mini",  # upgrade from 3.5, much better reasoning
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

In [26]:
question = "What is the fastest 5K time I've recently run?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
    max_tokens=200
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the fastest 5K time I've recently run?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_H1hDbvL1iehOSxqNn9MKTtxm)
 Call ID: call_H1hDbvL1iehOSxqNn9MKTtxm
  Args:
    query: SELECT start_date, distance, duration, pace FROM runs WHERE distance BETWEEN 4.8 AND 5.2 ORDER BY duration LIMIT 5
================================= Tool Message =================================
Name: execute_sql

[('2026-03-13T21:08:29Z', 5.0106, 19.116666666666667, 3.815245013903857), ('2026-03-27T21:03:18Z', 5.019699999999999, 19.533333333333335, 3.891334807525019), ('2026-03-05T06:32:52Z', 5.0234, 25.75, 5.12601027192738), ('2026-02-27T21:03:12Z', 4.980899999999999, 26.033333333333335, 5.226632402444004), ('2026-03-29T19:55:58Z', 5.1475, 27.4, 5.322972316658572)]
================================== Ai Message ==================================

Your fastest rec

In [28]:
question = "What is the longest distance I have recently run?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
    max_tokens=200
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the longest distance I have recently run?
================================== Ai Message ==================================
Tool Calls:
  execute_sql (call_pp96QzIi0YcPbuWE3JUTjSRJ)
 Call ID: call_pp96QzIi0YcPbuWE3JUTjSRJ
  Args:
    query: SELECT distance, start_date FROM runs ORDER BY distance DESC LIMIT 1
================================= Tool Message =================================
Name: execute_sql

[(17.391099999999998, '2026-03-28T21:06:47Z')]
================================== Ai Message ==================================

The longest distance you have recently run is approximately 17.39 km, which was on March 28, 2026.


In [35]:
question = "Is my fitness improving?"

for step in agent.stream(
    {"messages": question},
    context=RuntimeContext(db=db),
    stream_mode="values",
    max_tokens=200
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Is my fitness improving?
================================== Ai Message ==================================

To determine if your fitness is improving, we can look at several factors from your running activities, such as:

1. **Pace Over Time**: An improvement in pace (min/km) indicates better fitness.
2. **Distance and Duration**: Increased distance or duration may imply better endurance.
3. **Heart Rate Zones**: Monitoring time spent in various heart rate zones can reveal fitness gains.

I will need to review your recent runs to assess these aspects. How many of your most recent runs would you like me to analyze? A common approach is to look at the last 5 runs. Shall I proceed with that?
